# Code 7: Combine All Datasets for Model Building

## Purpose
Merge all processed datasets to create the **final modeling dataset** as a parquet file.

**This is pure data preparation** - no NaN handling, scaling, or encoding (handled in Code 8a/8b based on `feature_reference.csv`).

## What This Code Does
1. Load all input files from Google Drive
2. Merge features with conviction labels (on Date + Ticker)
3. Add cluster assignments (on Ticker)
4. Add own cluster returns (on Date + cluster)
5. **NEW:** Add counter-cluster returns via lookup
6. **NEW:** Add mapped commodity returns with **forward-fill** for date alignment
7. Create 12 derived features (RS_vs_Cluster, Cluster_vs_Counter, Commodity_Alignment)
8. Add `conviction_label_numeric` encoding
9. Convert float64 to float32 for memory efficiency
10. Save as `dataset_all.parquet`
11. Auto-download outputs to local Downloads folder

## ⚠️ Annual Cluster Refresh (important)
Clusters are refreshed **annually** (Code 4 v6). Membership is formed on Dec 31 of year Y-1 and applied to all of year Y. Every cluster join is keyed on `applicable_year = year(date)` to prevent look-ahead. A stock can belong to different clusters in different years.


## Input and Output Files

### INPUT FILES (read from Google Drive)
Expected location: `/content/drive/MyDrive/Colab Notebooks/MastersResearch/`

| # | Filename | Source | Description |
|---|----------|--------|-------------|
| 1 | `data_features.csv` | Code 3 V4 | ~175 stock features (returns, volatility, technical, calendar, etc.) |
| 2 | `clusters.csv` | Code 4 v6 | Per `(ticker, applicable_year)`: cluster, counter_cluster, mapped_commodity, correlations, commodity_direction |
| 3 | `cluster_returns.csv` | Code 5 v2 | ~52 cluster metrics per `(applicable_year, cluster, date)` |
| 4 | `commodity_returns.csv` | Code 5 v2 | ~43 commodity metrics per `(commodity, date)` (no applicable_year) |
| 5 | `data_exit_all.csv` | Code 6 | Conviction labels (only `entry_date`, `Ticker`, `conviction_label` loaded) |

### OUTPUT FILES (auto-downloaded to Downloads folder)

| # | Filename | Description | Size (est.) |
|---|----------|-------------|-------------|
| 1 | `dataset_all.parquet` | **MAIN OUTPUT** - Final modeling dataset (~347 cols, ~1.9M rows) | ~800 MB |
| 2 | `merge_summary.csv` | Documentation of all merge operations | <1 KB |
| 3 | `dataset_all_columns.csv` | Column list with dtype & null counts | ~30 KB |

### CONFIGURATION (set in Section 1)
- `USE_FLOAT32`: Memory optimization (default: True, saves ~50% RAM)
- `AUTO_DOWNLOAD`: Auto-download outputs to local Downloads (default: True)

## Section 1: Configuration

In [1]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Google Drive base path
GDRIVE_BASE = '/content/drive/MyDrive/masters'

# Input file names
FEATURES_FILE = 'data_features.parquet'
CLUSTERS_FILE = 'clusters.csv'
CLUSTER_RETURNS_FILE = 'cluster_returns.csv'
COMMODITY_RETURNS_FILE = 'commodity_returns.csv'
EXIT_FILE = 'data_exit_all.csv'

# Output file names
OUTPUT_DATASET = 'dataset_all.parquet'
OUTPUT_SUMMARY = 'merge_summary.csv'
OUTPUT_COLUMNS = 'dataset_all_columns.csv'

# Options
USE_FLOAT32 = True       # Convert float64 -> float32 (saves ~50% memory)
AUTO_DOWNLOAD = True     # Auto-download outputs to local Downloads folder

# Conviction label encoding (ordinal)
LABEL_MAPPING = {
    'Ignore': 0,   # No conviction - skip
    'Low':    1,   # 10-20% annualized return
    'Medium': 2,   # 20-40% annualized return
    'High':   3,   # 40%+ annualized return
}

print('Configuration loaded')
print(f'  GDrive base: {GDRIVE_BASE}')
print(f'  USE_FLOAT32: {USE_FLOAT32}')
print(f'  AUTO_DOWNLOAD: {AUTO_DOWNLOAD}')
print(f'  Label mapping: {LABEL_MAPPING}')

Configuration loaded
  GDrive base: /content/drive/MyDrive/masters
  USE_FLOAT32: True
  AUTO_DOWNLOAD: True
  Label mapping: {'Ignore': 0, 'Low': 1, 'Medium': 2, 'High': 3}


## Section 2: Imports and Google Drive Mount

In [2]:
# Standard imports
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported')
print(f'pandas version: {pd.__version__}')
print(f'numpy version:  {np.__version__}')
print(f'Script run at:  {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

Libraries imported
pandas version: 2.2.2
numpy version:  2.0.2
Script run at:  2026-06-22 11:25:54


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GDrive folder exists
if not os.path.exists(GDRIVE_BASE):
    raise FileNotFoundError(f'GDrive folder not found: {GDRIVE_BASE}')
print(f'\nGDrive base path verified: {GDRIVE_BASE}')

# Show all input files exist
print('\nChecking input files:')
input_files = [FEATURES_FILE, CLUSTERS_FILE, CLUSTER_RETURNS_FILE,
               COMMODITY_RETURNS_FILE, EXIT_FILE]
all_present = True
for f in input_files:
    fpath = os.path.join(GDRIVE_BASE, f)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f'  [OK] {f:30s} {size_mb:>8.1f} MB')
    else:
        print(f'  [MISSING] {f}')
        all_present = False

if not all_present:
    raise FileNotFoundError('Some input files are missing - check paths')

Mounted at /content/drive

GDrive base path verified: /content/drive/MyDrive/masters

Checking input files:
  [OK] data_features.parquet            1303.9 MB
  [OK] clusters.csv                        0.5 MB
  [OK] cluster_returns.csv               208.0 MB
  [OK] commodity_returns.csv              68.6 MB
  [OK] data_exit_all.csv                 435.9 MB


## Section 3: Load All Input Files

### 3.1 Load `data_features.csv`

In [4]:
print('Loading data_features.parquet...')
fpath = os.path.join(GDRIVE_BASE, FEATURES_FILE)

df_features = pd.read_parquet(fpath)
df_features['Date'] = pd.to_datetime(df_features['Date'])

print(f'  Loaded shape: {df_features.shape}')
print(f'  Date range:   {df_features["Date"].min()} to {df_features["Date"].max()}')
print(f'  Tickers:      {df_features["Ticker"].nunique()}')
print(f'  Memory:       {df_features.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(f'\n  First 5 columns: {list(df_features.columns[:5])}')
print(f'  Last 5 columns:  {list(df_features.columns[-5:])}')

Loading data_features.parquet...
  Loaded shape: (2417660, 175)
  Date range:   2007-01-02 00:00:00 to 2026-06-12 00:00:00
  Tickers:      570
  Memory:       1955.1 MB

  First 5 columns: ['Date', 'Open', 'High', 'Low', 'Close']
  Last 5 columns:  ['DayOfWeek_sin', 'DayOfWeek_cos', 'Is_Month_End', 'Is_Quarter_End', 'Is_FY_End']


### 3.2 Load `clusters.csv`

In [5]:
print('Loading clusters.csv...')
fpath = os.path.join(GDRIVE_BASE, CLUSTERS_FILE)

# Read with encoding fallback
for enc in ['utf-8', 'latin-1', 'cp1252']:
    try:
        df_clusters = pd.read_csv(fpath, encoding=enc)
        print(f'  Encoding: {enc}')
        break
    except UnicodeDecodeError:
        continue

# Normalize column name: symbol -> Ticker
if 'symbol' in df_clusters.columns and 'Ticker' not in df_clusters.columns:
    df_clusters = df_clusters.rename(columns={'symbol': 'Ticker'})
    print('  Renamed symbol -> Ticker')

print(f'  Loaded shape: {df_clusters.shape}')
print(f'  Tickers:      {df_clusters["Ticker"].nunique()}')
print(f'  Columns:      {list(df_clusters.columns)}')

if 'cluster' in df_clusters.columns:
    print(f'  Unique clusters: {df_clusters["cluster"].nunique()}')
if 'counter_cluster' in df_clusters.columns:
    print(f'  Unique counter_clusters: {df_clusters["counter_cluster"].nunique()} (null: {df_clusters["counter_cluster"].isna().sum()})')
if 'mapped_commodity' in df_clusters.columns:
    print(f'  Unique mapped_commodities: {df_clusters["mapped_commodity"].nunique()} (null: {df_clusters["mapped_commodity"].isna().sum()})')
if 'commodity_direction' in df_clusters.columns:
    print(f'  commodity_direction values: {df_clusters["commodity_direction"].dropna().unique()[:5]}')

print('\nSample rows:')
print(df_clusters.head())

Loading clusters.csv...
  Encoding: utf-8
  Renamed symbol -> Ticker
  Loaded shape: (7766, 11)
  Tickers:      570
  Columns:      ['applicable_year', 'formation_date', 'sector', 'Ticker', 'cluster', 'counter_cluster', 'cluster_correlation', 'counter_cluster_correlation', 'mapped_commodity', 'commodity_correlation', 'commodity_direction']
  Unique clusters: 277
  Unique counter_clusters: 0 (null: 7766)
  Unique mapped_commodities: 11 (null: 7518)
  commodity_direction values: ['none' 'positive' 'negative']

Sample rows:
   applicable_year formation_date           sector     Ticker  \
0             2011     2010-12-31  Basic Materials        ACC   
1             2011     2010-12-31  Basic Materials  AMBUJACEM   
2             2011     2010-12-31  Basic Materials     GRASIM   
3             2011     2010-12-31  Basic Materials   INDIACEM   
4             2011     2010-12-31  Basic Materials   JKCEMENT   

              cluster  counter_cluster  cluster_correlation  \
0  BasicMaterials-A

### 3.3 Load `cluster_returns.csv`

In [6]:
print('Loading cluster_returns.csv...')
fpath = os.path.join(GDRIVE_BASE, CLUSTER_RETURNS_FILE)

for enc in ['utf-8', 'latin-1', 'cp1252']:
    try:
        df_cluster_returns = pd.read_csv(fpath, encoding=enc)
        print(f'  Encoding: {enc}')
        break
    except UnicodeDecodeError:
        continue

if 'date' in df_cluster_returns.columns and 'Date' not in df_cluster_returns.columns:
    df_cluster_returns = df_cluster_returns.rename(columns={'date': 'Date'})
    print('  Renamed date -> Date')

df_cluster_returns['Date'] = pd.to_datetime(df_cluster_returns['Date'])

print(f'  Loaded shape: {df_cluster_returns.shape}')
print(f'  Date range:   {df_cluster_returns["Date"].min()} to {df_cluster_returns["Date"].max()}')
if 'cluster' in df_cluster_returns.columns:
    print(f'  Unique clusters: {df_cluster_returns["cluster"].nunique()}')
print(f'  Memory:       {df_cluster_returns.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(f'\n  Columns ({len(df_cluster_returns.columns)} total):')
print(f'  First 10:  {list(df_cluster_returns.columns[:10])}')
print(f'  Last 10:   {list(df_cluster_returns.columns[-10:])}')

Loading cluster_returns.csv...
  Encoding: utf-8
  Renamed date -> Date
  Loaded shape: (206229, 56)
  Date range:   2011-01-03 00:00:00 to 2025-12-31 00:00:00
  Unique clusters: 277
  Memory:       99.8 MB

  Columns (56 total):
  First 10:  ['applicable_year', 'cluster', 'Date', 'num_stocks', 'return_1d', 'return_2d', 'return_3d', 'return_5d', 'return_10d', 'return_20d']
  Last 10:   ['distance_from_low_120d', 'excess_return_1d', 'excess_return_2d', 'excess_return_3d', 'excess_return_5d', 'excess_return_10d', 'excess_return_20d', 'excess_return_60d', 'excess_return_120d', 'excess_return_252d']


### 3.4 Load `commodity_returns.csv`

In [7]:
print('Loading commodity_returns.csv...')
fpath = os.path.join(GDRIVE_BASE, COMMODITY_RETURNS_FILE)

for enc in ['utf-8', 'latin-1', 'cp1252']:
    try:
        df_commodity_returns = pd.read_csv(fpath, encoding=enc)
        print(f'  Encoding: {enc}')
        break
    except UnicodeDecodeError:
        continue

if 'date' in df_commodity_returns.columns and 'Date' not in df_commodity_returns.columns:
    df_commodity_returns = df_commodity_returns.rename(columns={'date': 'Date'})
    print('  Renamed date -> Date')

df_commodity_returns['Date'] = pd.to_datetime(df_commodity_returns['Date'])

print(f'  Loaded shape: {df_commodity_returns.shape}')
print(f'  Date range:   {df_commodity_returns["Date"].min()} to {df_commodity_returns["Date"].max()}')
if 'commodity' in df_commodity_returns.columns:
    print(f'  Unique commodities: {df_commodity_returns["commodity"].nunique()}')
    print(f'  Commodities: {sorted(df_commodity_returns["commodity"].unique())}')
print(f'  Memory:       {df_commodity_returns.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(f'\n  Columns ({len(df_commodity_returns.columns)} total):')
print(f'  First 10:  {list(df_commodity_returns.columns[:10])}')

Loading commodity_returns.csv...
  Encoding: utf-8
  Renamed date -> Date
  Loaded shape: (86190, 45)
  Date range:   2007-01-01 00:00:00 to 2026-06-12 00:00:00
  Unique commodities: 17
  Commodities: ['ALUMINIUM', 'BRENT_CRUDE', 'COCOA', 'COFFEE', 'COPPER', 'CORN', 'COTTON', 'CRUDE_OIL_WTI', 'GOLD', 'NATURAL_GAS', 'PALLADIUM', 'PLATINUM', 'SILVER', 'SOYBEAN', 'SUGAR', 'USDINR', 'WHEAT']
  Memory:       33.5 MB

  Columns (45 total):
  First 10:  ['commodity', 'Date', 'return_1d', 'return_2d', 'return_3d', 'return_5d', 'return_10d', 'return_20d', 'return_60d', 'return_120d']


### 3.5 Load `data_exit_all.csv` - ONLY conviction_label

We only need 3 columns: entry_date, Ticker, conviction_label.

In [8]:
print('Loading data_exit_all.csv (ONLY conviction_label)...')
fpath = os.path.join(GDRIVE_BASE, EXIT_FILE)

# Peek at columns (with encoding fallback)
for enc in ['utf-8', 'latin-1', 'cp1252']:
    try:
        df_peek = pd.read_csv(fpath, nrows=1, encoding=enc)
        exit_encoding = enc
        break
    except UnicodeDecodeError:
        continue
available_cols = list(df_peek.columns)
print(f'  Available columns: {available_cols}')

# Detect actual column names
date_col = 'entry_date' if 'entry_date' in available_cols else ('Date' if 'Date' in available_cols else None)
ticker_col = 'Ticker' if 'Ticker' in available_cols else ('ticker' if 'ticker' in available_cols else 'symbol')
label_col = 'conviction_label' if 'conviction_label' in available_cols else 'Conviction_Label'

print(f'  Detected: date={date_col}, ticker={ticker_col}, label={label_col}')

df_exit = pd.read_csv(fpath, usecols=[date_col, ticker_col, label_col], encoding=exit_encoding)

rename_map = {}
if date_col != 'Date':
    rename_map[date_col] = 'Date'
if ticker_col != 'Ticker':
    rename_map[ticker_col] = 'Ticker'
if label_col != 'conviction_label':
    rename_map[label_col] = 'conviction_label'
if rename_map:
    df_exit = df_exit.rename(columns=rename_map)
    print(f'  Renamed: {rename_map}')

df_exit['Date'] = pd.to_datetime(df_exit['Date'])

print(f'\n  Loaded shape: {df_exit.shape}')
print(f'  Memory:       {df_exit.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(f'\n  Conviction label distribution:')
print(df_exit['conviction_label'].value_counts(dropna=False))

Loading data_exit_all.csv (ONLY conviction_label)...
  Available columns: ['Ticker', 'entry_date', 'entry_price_raw', 'entry_price_adjusted', 'atr_at_entry', 'atr_source', 'stop_loss_threshold_rupees', 'stop_loss_pct', 'exit_date', 'exit_price_raw', 'exit_price_adjusted', 'exit_reason', 'max_price_achieved', 'drawdown_at_exit_rupees', 'drawdown_at_exit_pct', 'days_held', 'transaction_cost_pct', 'raw_return_pct', 'annualized_return_pct', 'data_quality_flag', 'conviction_label']
  Detected: date=entry_date, ticker=Ticker, label=conviction_label
  Renamed: {'entry_date': 'Date'}

  Loaded shape: (2417660, 3)
  Memory:       273.2 MB

  Conviction label distribution:
conviction_label
Ignore    1521589
High       603804
Medium     183820
Low        107877
NaN           570
Name: count, dtype: int64


## Section 4: Step 1 - Merge Features with Conviction Label

**Strategy:** LEFT merge on Date + Ticker. Only conviction_label is added (1 column).

In [9]:
print('STEP 1: Adding conviction_label to features...')
print('-' * 70)

before_rows = len(df_features)
before_cols = len(df_features.columns)
print(f'Before: {before_rows:,} rows x {before_cols} cols')

df_merged = df_features.merge(
    df_exit,
    on=['Date', 'Ticker'],
    how='left'
)

after_rows = len(df_merged)
after_cols = len(df_merged.columns)
print(f'After:  {after_rows:,} rows x {after_cols} cols')
print(f'Added {after_cols - before_cols} column(s)')

# Check label coverage
null_labels = df_merged['conviction_label'].isna().sum()
label_pct = (1 - null_labels / after_rows) * 100
print(f'\nConviction label coverage: {after_rows - null_labels:,} / {after_rows:,} ({label_pct:.1f}%)')
print(f'Null labels (expected for recent dates): {null_labels:,}')

print(f'\nLabel distribution (non-null):')
print(df_merged['conviction_label'].value_counts())

# Free memory
del df_exit, df_features
import gc; gc.collect()
print(f'\nMemory after merge: {df_merged.memory_usage(deep=True).sum()/1024**2:.1f} MB')

STEP 1: Adding conviction_label to features...
----------------------------------------------------------------------
Before: 2,417,660 rows x 175 cols
After:  2,417,660 rows x 176 cols
Added 1 column(s)

Conviction label coverage: 2,417,090 / 2,417,660 (100.0%)
Null labels (expected for recent dates): 570

Label distribution (non-null):
conviction_label
Ignore    1521589
High       603804
Medium     183820
Low        107877
Name: count, dtype: int64

Memory after merge: 2080.5 MB


## Section 5: Step 2 - Add Cluster Info

**Strategy:** LEFT merge on Ticker. Adds cluster, counter_cluster, mapped_commodity, correlations.

**Note:** Some columns may overlap with Code 3 (e.g., `sector`). We use suffix `_clust` and then drop redundant ones.

In [10]:
print('STEP 2: Adding cluster info (annual-refresh aware)...')
print('-' * 70)

before_cols = len(df_merged.columns)
print(f'Before: {len(df_merged):,} rows x {before_cols} cols')

# CRITICAL: clusters.csv now has one row per (Ticker, applicable_year).
# We MUST join on applicable_year too, or each stock-date row would match
# ~15 yearly assignments -> cartesian row explosion.
df_merged['applicable_year'] = df_merged['Date'].dt.year
print(f'Added applicable_year (= year of Date) for annual cluster matching')

# Verify clusters.csv has applicable_year
if 'applicable_year' not in df_clusters.columns:
    raise ValueError('clusters.csv missing applicable_year - is this Code 4 v6 output?')

# Track row count to guard against explosion
rows_before_merge = len(df_merged)

# Identify overlapping (non-key) columns -> they get _clust suffix
overlap = (set(df_clusters.columns) & set(df_merged.columns)) - {'Ticker', 'applicable_year'}
if overlap:
    print(f'Overlapping columns will get _clust suffix: {overlap}')

df_merged = df_merged.merge(
    df_clusters,
    on=['Ticker', 'applicable_year'],
    how='left',
    suffixes=('', '_clust')
)

# ROW-COUNT GUARD: left join on a key unique in df_clusters must not add rows
if len(df_merged) != rows_before_merge:
    raise ValueError(
        f'Row count changed after clusters merge! '
        f'{rows_before_merge:,} -> {len(df_merged):,}. '
        f'Check for duplicate (Ticker, applicable_year) in clusters.csv.'
    )
print(f'Row-count guard passed (no explosion): {len(df_merged):,} rows')

# Drop columns we do not need / that duplicate Code 3:
#  - formation_date: Code 4 metadata, not a feature
#  - sector (lowercase from clusters.csv): Code 3 already provides 'Sector'
#  - any _clust suffixed duplicates
drop_cols = [c for c in df_merged.columns if c.endswith('_clust')]
for c in ['formation_date', 'sector']:
    if c in df_merged.columns:
        drop_cols.append(c)
if drop_cols:
    df_merged = df_merged.drop(columns=drop_cols)
    print(f'Dropped: {drop_cols}')

after_cols = len(df_merged.columns)
print(f'\nAfter:  {len(df_merged):,} rows x {after_cols} cols')
print(f'Added {after_cols - before_cols} column(s)')

# Cluster coverage
if 'cluster' in df_merged.columns:
    null_cluster = df_merged['cluster'].isna().sum()
    print(f'\nCluster coverage: {len(df_merged) - null_cluster:,} / {len(df_merged):,}')
    if null_cluster > 0:
        n_missing = df_merged[df_merged['cluster'].isna()]['Ticker'].nunique()
        print(f'  Stock-years without cluster (late listings etc.): {n_missing} tickers')

print(f'\nMemory after merge: {df_merged.memory_usage(deep=True).sum()/1024**2:.1f} MB')

STEP 2: Adding cluster info (annual-refresh aware)...
----------------------------------------------------------------------
Before: 2,417,660 rows x 176 cols
Added applicable_year (= year of Date) for annual cluster matching
Row-count guard passed (no explosion): 2,417,660 rows
Dropped: ['formation_date', 'sector']

After:  2,417,660 rows x 184 cols
Added 8 column(s)

Cluster coverage: 1,914,912 / 2,417,660
  Stock-years without cluster (late listings etc.): 570 tickers

Memory after merge: 2471.7 MB


## Section 6: Step 3 - Add Own Cluster Returns

**Strategy:** LEFT merge on Date + cluster. The cluster_returns.csv has columns already prefixed with `cluster_` (e.g., `cluster_return_5d`).

In [11]:
print('STEP 3: Adding own cluster returns (annual-refresh aware)...')
print('-' * 70)

before_cols = len(df_merged.columns)
print(f'Before: {len(df_merged):,} rows x {before_cols} cols')

if 'cluster' not in df_cluster_returns.columns:
    raise ValueError('cluster_returns.csv missing cluster column')
if 'applicable_year' not in df_cluster_returns.columns:
    raise ValueError('cluster_returns.csv missing applicable_year - is this Code 5 v2 output?')

# Prefix all FEATURE columns with cluster_. Exclude the three join keys
# (Date, cluster, applicable_year) so they stay clean for the merge.
key_cols = ['Date', 'cluster', 'applicable_year']
non_key_cols = [c for c in df_cluster_returns.columns if c not in key_cols]
non_prefixed = [c for c in non_key_cols if not c.startswith('cluster_')]
if non_prefixed:
    rename_map = {c: f'cluster_{c}' for c in non_prefixed}
    df_cluster_returns = df_cluster_returns.rename(columns=rename_map)
    print(f'Added cluster_ prefix to {len(non_prefixed)} feature columns')

rows_before_merge = len(df_merged)

# Join on (cluster, applicable_year, Date). applicable_year is explicit and
# self-documenting; date alone pins the year but this is safer.
df_merged = df_merged.merge(
    df_cluster_returns,
    on=['cluster', 'applicable_year', 'Date'],
    how='left'
)

if len(df_merged) != rows_before_merge:
    raise ValueError(
        f'Row count changed after cluster_returns merge! '
        f'{rows_before_merge:,} -> {len(df_merged):,}.'
    )
print(f'Row-count guard passed: {len(df_merged):,} rows')

after_cols = len(df_merged.columns)
print(f'\nAfter:  {len(df_merged):,} rows x {after_cols} cols')
print(f'Added {after_cols - before_cols} column(s)')

sample_col = next((c for c in df_merged.columns if c.startswith('cluster_return_')), None)
if sample_col:
    null_pct = df_merged[sample_col].isna().sum() / len(df_merged) * 100
    print(f'\nSample column {sample_col}: {null_pct:.1f}% null')

print(f'\nMemory after merge: {df_merged.memory_usage(deep=True).sum()/1024**2:.1f} MB')

STEP 3: Adding own cluster returns (annual-refresh aware)...
----------------------------------------------------------------------
Before: 2,417,660 rows x 184 cols
Added cluster_ prefix to 53 feature columns
Row-count guard passed: 2,417,660 rows

After:  2,417,660 rows x 237 cols
Added 53 column(s)

Sample column cluster_return_1d: 20.8% null

Memory after merge: 3449.3 MB


## Section 7: Step 4 - Add Counter-Cluster Returns (Lookup)

**Strategy:**
1. Take cluster_returns.csv
2. Rename `cluster` -> `counter_cluster` and all `cluster_X` columns -> `counter_X`
3. LEFT merge on Date + counter_cluster

**Note:** Stocks with `counter_cluster = None/NaN` will have null counter columns. We do NOT impute - that's Code 8's job.

In [12]:
print('STEP 4: Adding counter-cluster returns via lookup...')
print('-' * 70)
print('NOTE: counter_cluster is ~all "None" in Indian equities (documented')
print('      negative finding). These columns are expected to be all-NaN.')
print('-' * 70)

before_cols = len(df_merged.columns)
print(f'Before: {len(df_merged):,} rows x {before_cols} cols')

# Build counter version by renaming cluster_ -> counter_
df_counter = df_cluster_returns.copy()
rename_map = {}
for col in df_counter.columns:
    if col == 'cluster':
        rename_map[col] = 'counter_cluster'
    elif col.startswith('cluster_'):
        rename_map[col] = 'counter_' + col[len('cluster_'):]
df_counter = df_counter.rename(columns=rename_map)
print(f'Renamed {len(rename_map)} columns (cluster_ -> counter_)')

if 'counter_cluster' not in df_merged.columns:
    print('\nWARNING: counter_cluster not in df_merged - skipping counter merge')
else:
    # Align dtypes; convert both counter_cluster to string. 'nan'/'None' simply
    # won't match real clusters, so rows correctly get NaN from the LEFT merge.
    df_counter['Date'] = pd.to_datetime(df_counter['Date'])
    df_merged['counter_cluster'] = df_merged['counter_cluster'].astype(str)
    df_counter['counter_cluster'] = df_counter['counter_cluster'].astype(str)

    rows_before_merge = len(df_merged)

    # Join on (Date, applicable_year, counter_cluster). Including applicable_year
    # as a KEY (not a stray column) avoids the applicable_year_x/_y collision.
    df_merged = df_merged.merge(
        df_counter,
        on=['Date', 'applicable_year', 'counter_cluster'],
        how='left'
    )

    if len(df_merged) != rows_before_merge:
        raise ValueError(
            f'Row count changed after counter merge! '
            f'{rows_before_merge:,} -> {len(df_merged):,}.'
        )
    print(f'Row-count guard passed: {len(df_merged):,} rows')

after_cols = len(df_merged.columns)
print(f'\nAfter:  {len(df_merged):,} rows x {after_cols} cols')
print(f'Added {after_cols - before_cols} column(s)')

sample_col = next((c for c in df_merged.columns if c.startswith('counter_return_')), None)
if sample_col:
    null_pct = df_merged[sample_col].isna().sum() / len(df_merged) * 100
    print(f'\nSample column {sample_col}: {null_pct:.1f}% null (expected ~100%)')

del df_counter, df_cluster_returns
gc.collect()
print(f'\nMemory after merge: {df_merged.memory_usage(deep=True).sum()/1024**2:.1f} MB')

STEP 4: Adding counter-cluster returns via lookup...
----------------------------------------------------------------------
NOTE: counter_cluster is ~all "None" in Indian equities (documented
      negative finding). These columns are expected to be all-NaN.
----------------------------------------------------------------------
Before: 2,417,660 rows x 237 cols
Renamed 54 columns (cluster_ -> counter_)
Row-count guard passed: 2,417,660 rows

After:  2,417,660 rows x 290 cols
Added 53 column(s)

Sample column counter_return_1d: 100.0% null (expected ~100%)

Memory after merge: 4528.3 MB


## Section 8: Step 5 - Add Commodity Returns (with Forward-Fill)

**Why forward-fill?** Commodities trade on different schedules than NSE:
- Crude oil, gold, etc. trade globally with different holidays
- Some commodity dates may not match NSE trading dates
- We use last-known commodity value (forward-fill) for NSE dates without exact commodity match

**Strategy:**
1. Rename commodity_returns columns: `return_5d` -> `commodity_return_5d` (add prefix), `commodity` -> `mapped_commodity`
2. Use `pd.merge_asof` with `direction='backward'` and `by='mapped_commodity'`
3. This finds nearest commodity date <= NSE date, per commodity
4. Result: forward-filled commodity values aligned to NSE dates

In [13]:
print('STEP 5: Adding commodity returns with forward-fill...')
print('-' * 70)

before_cols = len(df_merged.columns)
print(f'Before: {len(df_merged):,} rows x {before_cols} cols')

# Verify required columns
if 'commodity' not in df_commodity_returns.columns:
    raise ValueError('commodity_returns.csv missing commodity column')
if 'mapped_commodity' not in df_merged.columns:
    raise ValueError('df_merged missing mapped_commodity column - check clusters.csv merge')

# Add commodity_ prefix to all non-key columns
non_key_cols = [c for c in df_commodity_returns.columns if c not in ['Date', 'commodity']]
rename_map = {c: f'commodity_{c}' for c in non_key_cols if not c.startswith('commodity_')}
df_commodity_returns = df_commodity_returns.rename(columns=rename_map)
print(f'Added commodity_ prefix to {len(rename_map)} columns')

# Rename commodity -> mapped_commodity to match df_merged key
df_commodity_returns = df_commodity_returns.rename(columns={'commodity': 'mapped_commodity'})

# CRITICAL: merge_asof requires sorted data on the merge key
df_merged_sorted = df_merged.sort_values('Date').reset_index(drop=True)
df_commodity_sorted = df_commodity_returns.sort_values('Date').reset_index(drop=True)

print('\nPerforming merge_asof (backward = forward-fill to NSE dates)...')
print('  This finds the most recent commodity row with date <= NSE date, per commodity')

df_merged = pd.merge_asof(
    df_merged_sorted,
    df_commodity_sorted,
    on='Date',
    by='mapped_commodity',
    direction='backward'      # backward = use last available commodity value (forward-fill)
)

after_cols = len(df_merged.columns)
print(f'\nAfter:  {len(df_merged):,} rows x {after_cols} cols')
print(f'Added {after_cols - before_cols} column(s)')

# Check commodity coverage
sample_col = next((c for c in df_merged.columns if c.startswith('commodity_return_')), None)
if sample_col:
    # Null because: (a) no mapped_commodity, OR (b) date earlier than any commodity data
    null_pct = df_merged[sample_col].isna().sum() / len(df_merged) * 100
    no_mapping_pct = df_merged['mapped_commodity'].isna().sum() / len(df_merged) * 100
    print(f'\nSample column {sample_col}:')
    print(f'  Null: {null_pct:.1f}% (no_mapping: {no_mapping_pct:.1f}%, rest = early dates)')

# Free memory
del df_commodity_returns, df_merged_sorted, df_commodity_sorted
gc.collect()
print(f'\nMemory after merge: {df_merged.memory_usage(deep=True).sum()/1024**2:.1f} MB')

STEP 5: Adding commodity returns with forward-fill...
----------------------------------------------------------------------
Before: 2,417,660 rows x 290 cols
Added commodity_ prefix to 43 columns

Performing merge_asof (backward = forward-fill to NSE dates)...
  This finds the most recent commodity row with date <= NSE date, per commodity

After:  2,417,660 rows x 333 cols
Added 43 column(s)

Sample column commodity_return_1d:
  Null: 97.5% (no_mapping: 97.5%, rest = early dates)

Memory after merge: 5321.5 MB


## Section 9: Step 6 - Create Derived Features

12 derived features computed from existing columns:

1. **RS_vs_Cluster_1d, 5d, 20d, 60d** = Return_Xd - cluster_return_Xd
2. **Cluster_vs_Counter_1d, 5d, 20d, 60d** = cluster_return_Xd - counter_return_Xd
3. **Commodity_Alignment_1d, 5d, 20d, 60d** = Return_Xd * commodity_direction * commodity_return_Xd

In [14]:
print('STEP 6: Creating derived cross-dataset features...')
print('-' * 70)

before_cols = len(df_merged.columns)
print(f'Before: {before_cols} cols')

HORIZONS = [1, 5, 20, 60]

# ----- 0. Convert commodity_direction text -> numeric (+1/-1) -----
print('\n  Converting commodity_direction text to numeric...')
if 'commodity_direction' in df_merged.columns:
    print(f'    Before - unique: {df_merged["commodity_direction"].dropna().unique()[:6]}')
    direction_map = {
        'positive': 1, 'negative': -1,
        'Positive': 1, 'Negative': -1,
        'POSITIVE': 1, 'NEGATIVE': -1,
        '1': 1, '-1': -1, '1.0': 1, '-1.0': -1,
        'none': np.nan, 'None': np.nan, 'NONE': np.nan,
    }
    # Map text values; anything unmapped (including already-numeric) handled below
    mapped = df_merged['commodity_direction'].map(direction_map)
    # If mapping produced all-NaN (e.g. already numeric), fall back to to_numeric
    if mapped.notna().sum() == 0:
        mapped = pd.to_numeric(df_merged['commodity_direction'], errors='coerce')
    df_merged['commodity_direction'] = mapped
    print(f'    After  - unique: {sorted(df_merged["commodity_direction"].dropna().unique())}')
    print(f'    non-null: {df_merged["commodity_direction"].notna().sum():,}')
else:
    print('    WARNING: commodity_direction column not found')

# ----- 1. RS_vs_Cluster (Stock - Cluster) -----
print('\n  Creating RS_vs_Cluster_X = Return_Xd - cluster_return_Xd')
for h in HORIZONS:
    stock_col   = f'Return_{h}d'
    cluster_col = f'cluster_return_{h}d'
    derived_col = f'RS_vs_Cluster_{h}d'
    if stock_col in df_merged.columns and cluster_col in df_merged.columns:
        df_merged[stock_col]   = pd.to_numeric(df_merged[stock_col],   errors='coerce')
        df_merged[cluster_col] = pd.to_numeric(df_merged[cluster_col], errors='coerce')
        df_merged[derived_col] = df_merged[stock_col] - df_merged[cluster_col]
        print(f'    [OK] {derived_col}  (non-null: {df_merged[derived_col].notna().sum():,})')
    else:
        missing = [c for c in [stock_col, cluster_col] if c not in df_merged.columns]
        print(f'    [SKIP] {derived_col}: missing {missing}')

# ----- 2. Cluster_vs_Counter (Own Cluster - Counter Cluster) -----
print('\n  Creating Cluster_vs_Counter_X = cluster_return_Xd - counter_return_Xd')
for h in HORIZONS:
    own_col     = f'cluster_return_{h}d'
    counter_col = f'counter_return_{h}d'
    derived_col = f'Cluster_vs_Counter_{h}d'
    if own_col in df_merged.columns and counter_col in df_merged.columns:
        df_merged[own_col]     = pd.to_numeric(df_merged[own_col],     errors='coerce')
        df_merged[counter_col] = pd.to_numeric(df_merged[counter_col], errors='coerce')
        df_merged[derived_col] = df_merged[own_col] - df_merged[counter_col]
        print(f'    [OK] {derived_col}  (non-null: {df_merged[derived_col].notna().sum():,})')
    else:
        missing = [c for c in [own_col, counter_col] if c not in df_merged.columns]
        print(f'    [SKIP] {derived_col}: missing {missing}')

# ----- 3. Commodity_Alignment (Stock * direction * Commodity) -----
print('\n  Creating Commodity_Alignment_X = Return_Xd * commodity_direction * commodity_return_Xd')
if 'commodity_direction' not in df_merged.columns:
    print('    WARNING: commodity_direction missing - cannot create alignment features')
else:
    for h in HORIZONS:
        stock_col   = f'Return_{h}d'
        comm_col    = f'commodity_return_{h}d'
        derived_col = f'Commodity_Alignment_{h}d'
        if stock_col in df_merged.columns and comm_col in df_merged.columns:
            df_merged[stock_col] = pd.to_numeric(df_merged[stock_col], errors='coerce')
            df_merged[comm_col]  = pd.to_numeric(df_merged[comm_col],  errors='coerce')
            df_merged[derived_col] = (
                df_merged[stock_col] *
                df_merged['commodity_direction'] *
                df_merged[comm_col]
            )
            print(f'    [OK] {derived_col}  (non-null: {df_merged[derived_col].notna().sum():,})')
        else:
            missing = [c for c in [stock_col, comm_col] if c not in df_merged.columns]
            print(f'    [SKIP] {derived_col}: missing {missing}')

after_cols = len(df_merged.columns)
print(f'\nAfter:  {after_cols} cols (added {after_cols - before_cols} derived features)')
print(f'Memory: {df_merged.memory_usage(deep=True).sum()/1024**2:.1f} MB')

STEP 6: Creating derived cross-dataset features...
----------------------------------------------------------------------
Before: 333 cols

  Converting commodity_direction text to numeric...
    Before - unique: ['none' 'positive' 'negative']
    After  - unique: [np.float64(-1.0), np.float64(1.0)]
    non-null: 61,121

  Creating RS_vs_Cluster_X = Return_Xd - cluster_return_Xd
    [OK] RS_vs_Cluster_1d  (non-null: 1,914,912)
    [OK] RS_vs_Cluster_5d  (non-null: 1,913,012)
    [OK] RS_vs_Cluster_20d  (non-null: 1,905,887)
    [OK] RS_vs_Cluster_60d  (non-null: 1,886,913)

  Creating Cluster_vs_Counter_X = cluster_return_Xd - counter_return_Xd
    [OK] Cluster_vs_Counter_1d  (non-null: 0)
    [OK] Cluster_vs_Counter_5d  (non-null: 0)
    [OK] Cluster_vs_Counter_20d  (non-null: 0)
    [OK] Cluster_vs_Counter_60d  (non-null: 0)

  Creating Commodity_Alignment_X = Return_Xd * commodity_direction * commodity_return_Xd
    [OK] Commodity_Alignment_1d  (non-null: 61,121)
    [OK] Commodity_

## Section 10: Step 7 - Add conviction_label_numeric

Encode the categorical conviction_label to numeric (ordinal):
- Ignore -> 0, Low -> 1, Medium -> 2, High -> 3
- Null labels stay null (will be filtered/handled in Code 8)

In [15]:
print('STEP 7: Adding conviction_label_numeric encoding...')
print('-' * 70)

print(f'Label mapping: {LABEL_MAPPING}')

df_merged['conviction_label_numeric'] = df_merged['conviction_label'].map(LABEL_MAPPING)

# Validate
print(f'\nEncoded distribution:')
print(df_merged['conviction_label_numeric'].value_counts(dropna=False).sort_index())

# Cross-check: any non-null label that didn't get encoded?
label_present = df_merged['conviction_label'].notna()
encoded_present = df_merged['conviction_label_numeric'].notna()
mismatch = (label_present & ~encoded_present).sum()
if mismatch > 0:
    unknown_labels = df_merged[label_present & ~encoded_present]['conviction_label'].unique()
    print(f'WARNING: {mismatch} rows have label but no numeric. Unknown labels: {unknown_labels}')
else:
    print(f'\n[OK] All non-null labels encoded successfully')

STEP 7: Adding conviction_label_numeric encoding...
----------------------------------------------------------------------
Label mapping: {'Ignore': 0, 'Low': 1, 'Medium': 2, 'High': 3}

Encoded distribution:
conviction_label_numeric
0.0    1521589
1.0     107877
2.0     183820
3.0     603804
NaN        570
Name: count, dtype: int64

[OK] All non-null labels encoded successfully


## Section 11: Step 8 - Convert to float32 (Memory Optimization)

Converts all float64 numerical columns to float32. **No information loss** for stock data (7 decimals is plenty).

In [16]:
print('STEP 8: Float dtype optimization...')
print('-' * 70)

if USE_FLOAT32:
    before_mem = df_merged.memory_usage(deep=True).sum() / 1024**2
    print(f'Before: {before_mem:.1f} MB')

    # Convert all float64 to float32
    float_cols = df_merged.select_dtypes(include=['float64']).columns
    print(f'\nConverting {len(float_cols)} float64 columns to float32...')

    for col in float_cols:
        df_merged[col] = df_merged[col].astype('float32')

    after_mem = df_merged.memory_usage(deep=True).sum() / 1024**2
    saved_pct = (1 - after_mem/before_mem) * 100
    print(f'\nAfter:  {after_mem:.1f} MB ({saved_pct:.1f}% memory saved)')
else:
    print('USE_FLOAT32 = False, keeping float64')

# Final dtype summary
print(f'\nFinal dtype counts:')
print(df_merged.dtypes.value_counts())

STEP 8: Float dtype optimization...
----------------------------------------------------------------------
Before: 5467.4 MB

Converting 168 float64 columns to float32...

After:  3918.0 MB (28.3% memory saved)

Final dtype counts:
float32           325
int8               11
object              7
datetime64[ns]      1
int16               1
int32               1
Name: count, dtype: int64


## Section 12: Step 9 - Final Validation

In [17]:
print('STEP 9: Final dataset validation...')
print('=' * 70)

# 1. Shape
print(f'\n1. Shape: {df_merged.shape}')
print(f'   Memory: {df_merged.memory_usage(deep=True).sum()/1024**2:.1f} MB')

# 2. Duplicates
dups = df_merged.duplicated(subset=['Date', 'Ticker']).sum()
print(f'\n2. Duplicate (Date, Ticker): {dups}')

# 3. Date range
print(f'\n3. Date range: {df_merged["Date"].min()} to {df_merged["Date"].max()}')
print(f'   Unique dates: {df_merged["Date"].nunique():,}')
print(f'   Unique tickers: {df_merged["Ticker"].nunique()}')

# 4. Conviction label
print(f'\n4. Conviction label distribution:')
print(df_merged['conviction_label_numeric'].value_counts(dropna=False).sort_index())

# 5. Column groups
print(f'\n5. Column categories:')
groups = {
    'cluster_': len([c for c in df_merged.columns if c.startswith('cluster_')]),
    'counter_': len([c for c in df_merged.columns if c.startswith('counter_')]),
    'commodity_': len([c for c in df_merged.columns if c.startswith('commodity_')]),
    'RS_vs_Cluster_': len([c for c in df_merged.columns if c.startswith('RS_vs_Cluster_')]),
    'Cluster_vs_Counter_': len([c for c in df_merged.columns if c.startswith('Cluster_vs_Counter_')]),
    'Commodity_Alignment_': len([c for c in df_merged.columns if c.startswith('Commodity_Alignment_')]),
}
for g, c in groups.items():
    print(f'   {g:25s} {c} columns')

# 6. Top missing columns
print(f'\n6. Top 10 columns by null count:')
nulls = df_merged.isna().sum().sort_values(ascending=False)
for col, n in nulls.head(10).items():
    pct = n/len(df_merged)*100
    print(f'   {col:50s} {n:>10,} ({pct:5.1f}%)')

print('\n[OK] Validation complete')

STEP 9: Final dataset validation...

1. Shape: (2417660, 346)
   Memory: 3918.0 MB

2. Duplicate (Date, Ticker): 0

3. Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
   Unique dates: 4,799
   Unique tickers: 570

4. Conviction label distribution:
conviction_label_numeric
0.0    1521589
1.0     107877
2.0     183820
3.0     603804
NaN        570
Name: count, dtype: int64

5. Column categories:
   cluster_                  54 columns
   counter_                  55 columns
   commodity_                45 columns
   RS_vs_Cluster_            4 columns
   Cluster_vs_Counter_       4 columns
   Commodity_Alignment_      4 columns

6. Top 10 columns by null count:
   Cluster_vs_Counter_60d                              2,417,660 (100.0%)
   counter_excess_return_60d                           2,417,660 (100.0%)
   counter_excess_return_120d                          2,417,660 (100.0%)
   counter_distance_from_low_120d                      2,417,660 (100.0%)
   counter_distance_from_high

## Section 13: Step 10 - Save Output Files

In [18]:
print('STEP 10: Saving output files locally...')
print('-' * 70)

# Drop the transient join key applicable_year before saving.
# It is redundant with Code 3's existing 'Year' column and is NOT in
# feature_reference.csv. Dropping keeps the output schema matching the contract.
if 'applicable_year' in df_merged.columns:
    df_merged = df_merged.drop(columns=['applicable_year'])
    print('Dropped transient join key: applicable_year (redundant with Year)')

# ----- 1. Main dataset as parquet -----
print(f'\n1. Saving {OUTPUT_DATASET}...')
df_merged.to_parquet(OUTPUT_DATASET, index=False, compression='snappy')
size_mb = os.path.getsize(OUTPUT_DATASET) / 1024**2
print(f'   Saved: {OUTPUT_DATASET} ({size_mb:.1f} MB)')

# ----- 2. Merge summary -----
print(f'\n2. Creating merge summary...')
merge_summary = pd.DataFrame([
    {'step': 1, 'operation': 'Loaded data_features.parquet (Code 3)'},
    {'step': 2, 'operation': 'Merged conviction_label from data_exit_all (on Date, Ticker)'},
    {'step': 3, 'operation': 'Merged clusters.csv on (Ticker, applicable_year) [annual refresh]'},
    {'step': 4, 'operation': 'Merged cluster_returns on (cluster, applicable_year, Date)'},
    {'step': 5, 'operation': 'Merged counter-cluster returns on (Date, applicable_year, counter_cluster) [all-NaN expected]'},
    {'step': 6, 'operation': 'Merged commodity returns via merge_asof backward (forward-fill to NSE dates)'},
    {'step': 7, 'operation': 'Created 12 derived features (RS_vs_Cluster, Cluster_vs_Counter, Commodity_Alignment)'},
    {'step': 8, 'operation': 'Added conviction_label_numeric (Ignore=0, Low=1, Medium=2, High=3)'},
    {'step': 9, 'operation': 'Converted float64 -> float32; dropped transient applicable_year'},
    {'step': 10, 'operation': f'Final dataset: {len(df_merged):,} rows x {len(df_merged.columns)} cols'},
])
merge_summary.to_csv(OUTPUT_SUMMARY, index=False)
print(f'   Saved: {OUTPUT_SUMMARY}')

# ----- 3. Column reference -----
print(f'\n3. Creating column reference...')
col_ref = pd.DataFrame({
    'position': range(1, len(df_merged.columns) + 1),
    'column_name': df_merged.columns,
    'dtype': df_merged.dtypes.values.astype(str),
    'non_null': df_merged.count().values,
    'null_count': df_merged.isna().sum().values,
    'null_pct': (df_merged.isna().sum() / len(df_merged) * 100).round(2).values
})
col_ref.to_csv(OUTPUT_COLUMNS, index=False)
print(f'   Saved: {OUTPUT_COLUMNS}')

print(f'\n[OK] All output files saved to /content/ (Colab working dir)')

STEP 10: Saving output files locally...
----------------------------------------------------------------------
Dropped transient join key: applicable_year (redundant with Year)

1. Saving dataset_all.parquet...
   Saved: dataset_all.parquet (1495.5 MB)

2. Creating merge summary...
   Saved: merge_summary.csv

3. Creating column reference...
   Saved: dataset_all_columns.csv

[OK] All output files saved to /content/ (Colab working dir)


## Section 14: Step 11 - Auto-Download to Local Downloads Folder

In [19]:
print('STEP 11: Auto-downloading output files...')
print('-' * 70)

if AUTO_DOWNLOAD:
    from google.colab import files

    outputs_to_download = [OUTPUT_SUMMARY, OUTPUT_COLUMNS, OUTPUT_DATASET]

    print('\nNote: dataset_all.parquet is large (~800 MB). Download may take several minutes.')
    print('Downloads happen one at a time. Browser may prompt for each.\n')

    for fname in outputs_to_download:
        if os.path.exists(fname):
            size_mb = os.path.getsize(fname) / 1024**2
            print(f'Downloading {fname} ({size_mb:.1f} MB)...')
            files.download(fname)
        else:
            print(f'SKIP: {fname} not found')

    print('\n[OK] All downloads triggered')
    print('Files will appear in your browser Downloads folder')
else:
    print('AUTO_DOWNLOAD = False. Files remain in /content/ on Colab.')
    print('Manually download via Files panel on left sidebar.')

STEP 11: Auto-downloading output files...
----------------------------------------------------------------------

Note: dataset_all.parquet is large (~800 MB). Download may take several minutes.
Downloads happen one at a time. Browser may prompt for each.



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


[OK] All downloads triggered
Files will appear in your browser Downloads folder


## Section 15: Final Summary

In [20]:
print('=' * 70)
print('CODE 7 COMPLETE')
print('=' * 70)

print(f'\nDataset Summary:')
print(f'  Rows:    {len(df_merged):,}')
print(f'  Columns: {len(df_merged.columns)}')
print(f'  Memory:  {df_merged.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(f'  Date range: {df_merged["Date"].min().date()} to {df_merged["Date"].max().date()}')
print(f'  Tickers: {df_merged["Ticker"].nunique()}')

print(f'\nOutput files (in /content/ and auto-downloaded):')
for f in [OUTPUT_DATASET, OUTPUT_SUMMARY, OUTPUT_COLUMNS]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024**2
        print(f'  [OK] {f:35s} {size:>8.1f} MB')

print(f'\nNext steps:')
print(f'  1. Upload dataset_all.parquet to GDrive (if needed for Code 8a/8b)')
print(f'  2. Use feature_reference.csv to drive Code 8 preprocessing decisions')
print(f'  3. Code 8a: XGBoost model with all relevant features')
print(f'  4. Code 8b: LSTM model using only lstm_relevant=Yes features')

print(f'\nFinished at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

CODE 7 COMPLETE

Dataset Summary:
  Rows:    2,417,660
  Columns: 345
  Memory:  3908.7 MB
  Date range: 2007-01-02 to 2026-06-12
  Tickers: 570

Output files (in /content/ and auto-downloaded):
  [OK] dataset_all.parquet                   1495.5 MB
  [OK] merge_summary.csv                        0.0 MB
  [OK] dataset_all_columns.csv                  0.0 MB

Next steps:
  1. Upload dataset_all.parquet to GDrive (if needed for Code 8a/8b)
  2. Use feature_reference.csv to drive Code 8 preprocessing decisions
  3. Code 8a: XGBoost model with all relevant features
  4. Code 8b: LSTM model using only lstm_relevant=Yes features

Finished at: 2026-06-22 11:29:07
